In [45]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/rohitsahoo/sales-forecasting/train.csv


In [46]:
import pandas as pd
import sqlite3

df = pd.read_csv("/kaggle/input/datasets/rohitsahoo/sales-forecasting/train.csv", encoding='latin-1')
print(df.shape)
df.head()

(9800, 18)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [47]:
# Create SQLite database connection
conn = sqlite3.connect("sales_warehouse.db")
cursor = conn.cursor()

print("Database created successfully!")

Database created successfully!


In [48]:
cursor.executescript("""
    CREATE TABLE IF NOT EXISTS dim_customers (
        customer_id TEXT PRIMARY KEY,
        customer_name TEXT,
        segment TEXT
    );

    CREATE TABLE IF NOT EXISTS dim_products (
        product_id TEXT PRIMARY KEY,
        product_name TEXT,
        category TEXT,
        sub_category TEXT
    );

    CREATE TABLE IF NOT EXISTS dim_location (
        location_id INTEGER PRIMARY KEY AUTOINCREMENT,
        country TEXT,
        city TEXT,
        state TEXT,
        postal_code TEXT,
        region TEXT
    );

    CREATE TABLE IF NOT EXISTS dim_date (
        date_id INTEGER PRIMARY KEY,
        order_date TEXT,
        year INTEGER,
        month INTEGER,
        quarter INTEGER
    );

    CREATE TABLE IF NOT EXISTS fact_sales (
        row_id INTEGER PRIMARY KEY,
        order_id TEXT,
        customer_id TEXT,
        product_id TEXT,
        location_id INTEGER,
        date_id INTEGER,
        sales REAL,
        quantity INTEGER,
        discount REAL,
        profit REAL
    );
""")

conn.commit()
print("All tables created!")

All tables created!


In [49]:
# dim_customers
customers = df[['Customer ID', 'Customer Name', 'Segment']].drop_duplicates()
customers.columns = ['customer_id', 'customer_name', 'segment']
customers.to_sql('dim_customers', conn, if_exists='replace', index=False)

# dim_products
products = df[['Product ID', 'Product Name', 'Category', 'Sub-Category']].drop_duplicates()
products.columns = ['product_id', 'product_name', 'category', 'sub_category']
products.to_sql('dim_products', conn, if_exists='replace', index=False)

# dim_location
location = df[['Country', 'City', 'State', 'Postal Code', 'Region']].drop_duplicates().reset_index(drop=True)
location.columns = ['country', 'city', 'state', 'postal_code', 'region']
location['location_id'] = location.index + 1
location.to_sql('dim_location', conn, if_exists='replace', index=False)

# dim_date
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
dates = df[['Order Date']].drop_duplicates().reset_index(drop=True)
dates.columns = ['order_date']
dates['date_id'] = dates.index + 1
dates['year'] = dates['order_date'].dt.year
dates['month'] = dates['order_date'].dt.month
dates['quarter'] = dates['order_date'].dt.quarter
dates.to_sql('dim_date', conn, if_exists='replace', index=False)

print("Dimension tables populated!")

Dimension tables populated!


In [50]:
# 1. Row count check
cursor.execute("SELECT COUNT(*) FROM fact_sales")
print("Fact rows:", cursor.fetchone()[0])

# 2. Sales by region
cursor.execute("""
    SELECT l.region, ROUND(SUM(f.sales), 2) as total_sales
    FROM fact_sales f
    JOIN dim_location l ON f.location_id = l.location_id
    GROUP BY l.region
    ORDER BY total_sales DESC
""")
print("\nSales by Region:")
for row in cursor.fetchall():
    print(row)

# 3. Sales by year
cursor.execute("""
    SELECT d.year, ROUND(SUM(f.sales), 2) as total_sales
    FROM fact_sales f
    JOIN dim_date d ON f.date_id = d.date_id
    GROUP BY d.year
    ORDER BY d.year
""")
print("\nSales by Year:")
for row in cursor.fetchall():
    print(row)

# 4. Top 5 customers
cursor.execute("""
    SELECT c.customer_name, ROUND(SUM(f.sales), 2) as total_sales
    FROM fact_sales f
    JOIN dim_customers c ON f.customer_id = c.customer_id
    GROUP BY c.customer_name
    ORDER BY total_sales DESC
    LIMIT 5
""")
print("\nTop 5 Customers:")
for row in cursor.fetchall():
    print(row)

# 5. Null check
cursor.execute("SELECT COUNT(*) FROM fact_sales WHERE sales IS NULL")
print("\nNull sales count:", cursor.fetchone()[0])

Fact rows: 9800

Sales by Region:
('West', 710219.68)
('East', 669518.73)
('Central', 492646.91)
('South', 389151.46)

Sales by Year:
(2015, 479856.21)
(2016, 459436.01)
(2017, 600192.55)
(2018, 722052.02)

Top 5 Customers:
('Sean Miller', 25043.05)
('Tamara Chand', 19052.22)
('Raymond Buch', 15117.34)
('Tom Ashbrook', 14595.62)
('Adrian Barton', 14473.57)

Null sales count: 0


In [51]:
# Merge to get location_id and date_id
df_merged = df.merge(location, left_on=['Country', 'City', 'State', 'Postal Code', 'Region'],
                                right_on=['country', 'city', 'state', 'postal_code', 'region'], how='left')
df_merged = df_merged.merge(dates, left_on='Order Date', right_on='order_date', how='left')

# Build fact table
fact = df_merged[['Row ID', 'Order ID', 'Customer ID', 'Product ID', 'location_id', 'date_id', 'Sales', 'Quantity', 'Discount', 'Profit']]
fact.columns = ['row_id', 'order_id', 'customer_id', 'product_id', 'location_id', 'date_id', 'sales', 'quantity', 'discount', 'profit']

fact.to_sql('fact_sales', conn, if_exists='replace', index=False)
conn.commit()

print(df_merged.columns.tolist())

KeyError: "['Quantity', 'Discount', 'Profit'] not in index"

In [ ]:
print(df_merged.columns.tolist())

In [ ]:
fact = df_merged[['Row ID', 'Order ID', 'Customer ID', 'Product ID', 'location_id', 'date_id', 'Sales']]
fact.columns = ['row_id', 'order_id', 'customer_id', 'product_id', 'location_id', 'date_id', 'sales']

fact.to_sql('fact_sales', conn, if_exists='replace', index=False)
conn.commit()

print(f"Fact table loaded! {len(fact)} rows inserted.")

In [ ]:
import datetime

cursor.execute("""
    CREATE TABLE IF NOT EXISTS audit_log (
        audit_id INTEGER PRIMARY KEY AUTOINCREMENT,
        run_timestamp TEXT,
        fact_row_count INTEGER,
        null_sales_count INTEGER,
        total_sales REAL
    )
""")

def run_audit():
    timestamp = datetime.datetime.now().isoformat()
    row_count = cursor.execute("SELECT COUNT(*) FROM fact_sales").fetchone()[0]
    null_count = cursor.execute("SELECT COUNT(*) FROM fact_sales WHERE sales IS NULL").fetchone()[0]
    total_sales = cursor.execute("SELECT ROUND(SUM(sales),2) FROM fact_sales").fetchone()[0]
    
    cursor.execute("INSERT INTO audit_log (run_timestamp, fact_row_count, null_sales_count, total_sales) VALUES (?,?,?,?)",
                   (timestamp, row_count, null_count, total_sales))
    conn.commit()
    print(f"Audit logged at {timestamp} | Rows: {row_count} | Nulls: {null_count} | Total Sales: {total_sales}")

run_audit()

In [ ]:
audit_df = pd.read_sql("SELECT * FROM audit_log", conn)
audit_df.to_csv("audit_log.csv", index=False)
print("Audit log saved!")
audit_df

In [ ]:
audit_df = pd.read_sql("SELECT * FROM audit_log", conn)
audit_df.to_csv("audit_log.csv", index=False)
print("Audit log saved!")
audit_df